In [0]:
schema = 'fifa_bi_dev.silver_schema'

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df = spark.read.table('fifa_bi_dev.bronze_schema.worldcups')


result = df.withColumn('year', col('year').cast('int'))
result = result.withColumn('tournament_name', concat_ws(' ', lit('FIFA World Cup'), col('year')))
result = result.withColumn('host_country', col('country')).drop('country')
result = result.withColumn('total_goals', col('goalsscored').cast('int')).drop('goalsscored')
result = result.withColumn('qualified_team', col('qualifiedteams').cast('int')).drop('qualifiedteams')
result = result.withColumn('total_matches', col('matchesplayed').cast('int')).drop('matchesplayed')



result = result.withColumn(
    "final_attendance",
    (
        regexp_replace(col("attendance"), r"\.", "").cast("long")
        *
        when(~col("attendance").contains("."),1000)
        .when(length(substring_index(col("attendance"), ".", -1)) == 0,1000)
        .when(length(substring_index(col("attendance"), ".", -1)) == 1,100)
        .when(length(substring_index(col("attendance"), ".", -1)) == 2,10)
        .otherwise(1)
    )
)

result.show()

In [0]:
team = spark.read.table('fifa_bi_dev.silver_schema.seed_team_codes')

#team_country -


final = result.alias('tournament').join(team.alias('t'), col('winner') == col('t.team_name'), 'left')\
                .withColumn('winner', col('country_name')).drop('team_code', 'team_name', 'country_name')

final = final.alias('tournament').join(team.alias('t'), col('runners_up') == col('t.team_name'), 'left')\
                .withColumn('runners_up', col('country_name')).drop('team_code', 'team_name', 'country_name')
final = final.alias('tournament').join(team.alias('t'), col('third') == col('t.team_name'), 'left')\
                .withColumn('third', col('country_name')).drop('team_code', 'team_name', 'country_name')
final = final.alias('tournament').join(team.alias('t'), col('fourth') == col('t.team_name'), 'left')\
                .withColumn('fourth', col('country_name')).drop('team_code', 'team_name', 'country_name')




final.show()

In [0]:
result.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f'{schema}.silver_worldcups')
print(f'table_name: worldcups is updated')